In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )

base = "../data/results/";

In [ ]:
# System size.
N = 100

# Social parameters.
ρ = 0.01;  β = 0.1

# System parameters.
μlist = [0.1, 1.0, 10.0];
ℓlist = .√(N./μlist)
Nμ = length( μlist )

# Activity transition variables.
η = 0.0002;  γ = 0.01;  τ = 750.0

# Movement variables.
ξ = 1.0;  λ = 0.50;  ϕ = 0.25
s = 5.869067

# Sensing area parameter.
α = (1/4)π

# Generate parameter variable.
nondim = Nondim(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s, α=α )
println( "nondim. parameters: ", nondim )

# Create folderlist.
folderlist = [findfolder( N, μ, nondim; base=base ) for μ ∈ μlist]

In [ ]:
# Import position data.
M = 50
xdatadata = [[readdlm( folder*"sim/x-state_m-$(m).txt" ) for m ∈ 1:M] for folder ∈ folderlist]
ydatadata = [[readdlm( folder*"sim/y-state_m-$(m).txt" ) for m ∈ 1:M] for folder ∈ folderlist]

In [ ]:
# Compute pair-wise distances.
Tdist = 201
Ddatalist = [[[pairwisedist( ℓ, xdata[t,:], ydata[t,:]; n=100 )
    for t ∈ 1:Tdist] for (xdata, ydata) ∈ zip( xdatalist, ydatalist )]
    for (ℓ, xdatalist, ydatalist) ∈ zip( ℓlist, xdatadata, ydatadata )]

# Build pair-wise distributions.
ddata = [[d for Dlist ∈ Ddata for D ∈ Dlist for d ∈ D]
    for Ddata ∈ Ddatalist]

# Compute the distribution describing the spread of agents.
dmax = maximum( maximum.( ddata ) )
Δd = 0.01;  dbins = 0:Δd:maximum( ℓlist )/√2
Hlist = [fit( Histogram, dlist, dbins ) for dlist ∈ ddata]
@assert dmax < maximum( ℓlist )/√2 "There exists a distance greater than the boundaries allow."

# Normalize distribution for plotting.
d̂list = dbins[1:end-1] .+ Δd/2
fdata = [H.weights/(Δd*sum( H.weights )) for H ∈ Hlist]
fmax = maximum( maximum.( fdata ) );

# Line describing the slope of p(d).
xdata = [0:Δd:ℓ/2 for ℓ ∈ ℓlist]
ydata = [(2π/ℓ^2).*xlist for (ℓ, xlist) ∈ zip( ℓlist, xdata )]

In [ ]:
colorlist = [:black, :cornflowerblue, :indianred]

# Plot the distribution of pair-waise distances.
plt = plot( size=(350,200), dpi=100, legend=:outerright )
plot!( plt; left_margin=2.5pt, bottom_margin=5pt, right_margin=2.5pt )

αlist = [1/5, 1/5, 1/5]
for i ∈ Nμ:-1:1
    # Plot the data.
    ilist = 0.0 .< fdata[i]
    scatter!( plt, d̂list[ilist], fdata[i][ilist];
        color=colorlist[i] == :black ? :gray : colorlist[i],
        alpha=αlist[i], marker=:circ, markerstrokewidth=0, label="" )
    plot!( plt, xdata[i], ydata[i]; color=colorlist[i], lw=2,
        label=latexstring( "μ=10^{$(round( defInt, log10( μlist[i] ) ))}" ) )
end

plot!( plt; xlims=10.0.^[-2,2], xscale=:log10 )
plot!( plt; ylims=10.0.^[-4,0], yscale=:log10 )
plot!( plt; xlabel="pairwise distance, "*L"d", ylabel="probability density\nof distance, "*L"p(d)" )
plot!( plt; title="(a)", titleposition=:left )

# saveplot( plt, figurefolder*"activity_pairwise-distance_N-$(N)_μ-$(μlist[1])-$(μlist[end]).png"; dpi=600 );